Pré-processamento

In [1]:
!pip install mediapipe opencv-python pandas numpy tqdm

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import multiprocessing
from collections import Counter
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed


/home/tomas/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [3]:
# =========================
# CONFIGURAÇÃO
# =========================

# Ajusta estes caminhos ao teu dataset real
TRAIN_DIR = "asl/train"
TEST_DIR = "asl/test"

OUTPUT_TRAIN_CSV = "outputs/train_landmarks.csv"
OUTPUT_TEST_CSV = "outputs/test_landmarks.csv"
OUTPUT_TRAIN_STATS = "outputs/train_stats.csv"
OUTPUT_TEST_STATS = "outputs/test_stats.csv"
OUTPUT_EXTRACTION_SUMMARY = "outputs/extraction_summary.json"

CLASSES = [chr(c) for c in range(ord('A'), ord('Z') + 1)] + ["space", "nothing", "del"]

# Tamanhos a testar por ordem
SIZES_PRIMARY = [256, 384, 512]
SIZES_SECONDARY = [384, 512]

# Coloca None para processar tudo
MAX_IMAGES_PER_CLASS = 7000  # None

# Número de processos paralelos (None = automático: número de CPUs)
NUM_WORKERS = None


In [4]:
# =========================
# FUNÇÕES AUXILIARES
# =========================

def get_border_median_color(image, border=10):
    h, w = image.shape[:2]
    border = max(1, min(border, h, w))

    top = image[:border, :, :]
    bottom = image[h-border:h, :, :]
    left = image[:, :border, :]
    right = image[:, w-border:w, :]

    pixels = np.concatenate([
        top.reshape(-1, 3),
        bottom.reshape(-1, 3),
        left.reshape(-1, 3),
        right.reshape(-1, 3)
    ], axis=0)

    median_color = np.median(pixels, axis=0)
    return tuple(int(x) for x in median_color)


def resize_and_pad(image, target_size=256):
    h, w = image.shape[:2]

    scale = target_size / max(h, w)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    resized = cv2.resize(image, (new_w, new_h), interpolation=interp)

    delta_w = target_size - new_w
    delta_h = target_size - new_h

    top = delta_h // 2
    bottom = delta_h - top
    left = delta_w // 2
    right = delta_w - left

    pad_color = get_border_median_color(image)

    padded = cv2.copyMakeBorder(
        resized,
        top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT,
        value=pad_color
    )
    return padded


def apply_clahe_bgr(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l2 = clahe.apply(l)
    lab2 = cv2.merge((l2, a, b))
    return cv2.cvtColor(lab2, cv2.COLOR_LAB2BGR)


def sharpen_image(image):
    kernel = np.array([[0, -1, 0],
                       [-1, 5, -1],
                       [0, -1, 0]], dtype=np.float32)
    return cv2.filter2D(image, -1, kernel)


def estimate_hand_bbox_from_landmarks(landmarks, image_shape, margin=0.20):
    h, w = image_shape[:2]
    xs = [lm.x for lm in landmarks]
    ys = [lm.y for lm in landmarks]

    min_x = max(0.0, min(xs) - margin)
    max_x = min(1.0, max(xs) + margin)
    min_y = max(0.0, min(ys) - margin)
    max_y = min(1.0, max(ys) + margin)

    x1 = int(min_x * w)
    y1 = int(min_y * h)
    x2 = int(max_x * w)
    y2 = int(max_y * h)

    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def normalize_landmarks(lms):
    """Normaliza landmarks: coordenadas relativas ao pulso (63 valores)
    + distâncias euclidianas ao pulso (21 valores) = 84 features no total."""
    # 1. Coordenadas Relativas ao Pulso (Ponto 0)
    wx, wy, wz = lms[0].x, lms[0].y, lms[0].z

    # 2. Cálculo da Escala Robusta
    base_ids = [5, 9, 13, 17]
    dists_base = []
    for idx in base_ids:
        dx = lms[idx].x - wx
        dy = lms[idx].y - wy
        dz = lms[idx].z - wz
        dists_base.append((dx**2 + dy**2 + dz**2) ** 0.5)

    scale = float(np.mean(dists_base))
    if scale < 1e-6:
        return None

    features = []

    # 3. Atributos de Coordenadas (63 valores)
    for lm in lms:
        features.extend([
            (lm.x - wx) / scale,
            (lm.y - wy) / scale,
            (lm.z - wz) / scale
        ])

    # 4. Distâncias Euclidianas ao Pulso (21 valores)
    # Ajuda a distinguir M, N, S pela "compactação" da mão
    for lm in lms:
        d = ((lm.x - wx)**2 + (lm.y - wy)**2 + (lm.z - wz)**2)**0.5
        features.append(d / scale)

    # Verificação defensiva: garantir sempre 84 features
    assert len(features) == 84, f"Número de features inesperado: {len(features)} (esperado 84)"
    return features  # 84 valores


def run_hands_detector(image_bgr, detector):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return detector.process(image_rgb)


def detect_landmarks_with_retries(image_bgr, hands_primary, hands_relaxed):
    """Tenta detetar landmarks com várias estratégias em cascata.
    Pré-calcula CLAHE e Sharpen uma única vez para evitar recomputação."""
    # Pré-computar variantes da imagem uma única vez
    image_clahe = apply_clahe_bgr(image_bgr)
    image_sharp = sharpen_image(image_bgr)

    # 1) Original
    for size in SIZES_PRIMARY:
        candidate = resize_and_pad(image_bgr, target_size=size)
        results = run_hands_detector(candidate, hands_primary)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark, f"original_{size}_primary", candidate

    # 2) CLAHE
    for size in SIZES_PRIMARY:
        candidate = resize_and_pad(image_clahe, target_size=size)
        results = run_hands_detector(candidate, hands_primary)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark, f"clahe_{size}_primary", candidate

    # 3) Sharpen
    for size in SIZES_SECONDARY:
        candidate = resize_and_pad(image_sharp, target_size=size)
        results = run_hands_detector(candidate, hands_primary)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark, f"sharp_{size}_primary", candidate

    # 4) Detector mais permissivo na original
    for size in SIZES_PRIMARY:
        candidate = resize_and_pad(image_bgr, target_size=size)
        results = run_hands_detector(candidate, hands_relaxed)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark, f"original_{size}_relaxed", candidate

    # 5) Detector mais permissivo + CLAHE
    for size in SIZES_PRIMARY:
        candidate = resize_and_pad(image_clahe, target_size=size)
        results = run_hands_detector(candidate, hands_relaxed)
        if results.multi_hand_landmarks:
            return results.multi_hand_landmarks[0].landmark, f"clahe_{size}_relaxed", candidate

    return None, None, None


def crop_and_redetect(image_bgr, hands_primary, hands_relaxed):
    """Deteta landmarks, refina com crop se possível.
    O método inclui sufixo '_crop_failed' quando o crop não melhora o resultado."""
    landmarks, method, processed = detect_landmarks_with_retries(image_bgr, hands_primary, hands_relaxed)
    if landmarks is None:
        return None, None

    bbox = estimate_hand_bbox_from_landmarks(landmarks, processed.shape, margin=0.20)
    if bbox is None:
        return landmarks, method + "_no_bbox"

    x1, y1, x2, y2 = bbox
    crop = processed[y1:y2, x1:x2]
    if crop.size == 0:
        return landmarks, method + "_empty_crop"

    crop = resize_and_pad(crop, target_size=384)
    results = run_hands_detector(crop, hands_primary)
    if results.multi_hand_landmarks:
        return results.multi_hand_landmarks[0].landmark, method + "_crop_refined"

    # Crop falhou: retorna os landmarks originais mas regista explicitamente
    return landmarks, method + "_crop_failed"


def extract_hand_landmarks(image_path, hands_primary, hands_relaxed):
    """Função principal de extração. Recebe os detectors como argumento
    para ser compatível com paralelismo por processo."""
    image = cv2.imread(image_path)
    if image is None:
        return None, "read_error"

    landmarks, method = crop_and_redetect(image, hands_primary, hands_relaxed)
    if landmarks is None:
        return None, "no_hand_detected"

    features = normalize_landmarks(landmarks)
    if features is None:
        return None, "invalid_scale"

    return features, method


def build_columns():
    """Constrói os nomes das colunas: 63 coordenadas + 21 distâncias + label = 85 colunas."""
    cols = []
    for i in range(21):
        cols.extend([f"x{i}", f"y{i}", f"z{i}"])
    for i in range(21):
        cols.append(f"dist_{i}")
    cols.append("label")
    return cols


def collect_test_files(test_dir, max_per_class=None):
    """Recolhe ficheiros de teste. Suporta agora limite por classe (max_per_class),
    alinhado com o comportamento do treino."""
    rows = []

    flat_files = sorted(
        f for f in os.listdir(test_dir)
        if os.path.isfile(os.path.join(test_dir, f)) and f.lower().endswith((".jpg", ".jpeg", ".png"))
    )
    for fname in flat_files:
        label = os.path.splitext(fname)[0]
        if label.endswith("_test"):
            label = label[:-5]
        rows.append((os.path.join(test_dir, fname), label))

    subdirs = sorted(
        d for d in os.listdir(test_dir)
        if os.path.isdir(os.path.join(test_dir, d))
    )
    for label in subdirs:
        class_dir = os.path.join(test_dir, label)
        image_files = sorted(
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )
        if max_per_class is not None:
            image_files = image_files[:max_per_class]
        for fname in image_files:
            rows.append((os.path.join(class_dir, fname), label))

    return rows


# =========================
# FUNÇÃO WORKER PARA PARALELISMO
# =========================
# Cada processo cria os seus próprios detectors (MediaPipe não é thread-safe)
# Esta função é chamada por ProcessPoolExecutor e é executada num processo separado.

def _worker_init():
    """Inicializa os detectors MediaPipe no processo filho.
    Guardados em variáveis globais do processo para reutilização entre tarefas."""
    global _hands_primary, _hands_relaxed
    _hands_primary = mp.solutions.hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        model_complexity=1,
        min_detection_confidence=0.35
    )
    _hands_relaxed = mp.solutions.hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        model_complexity=1,
        min_detection_confidence=0.20
    )


def _worker_process_image(args):
    """Processa uma imagem num processo filho. Reutiliza os detectors inicializados."""
    img_path, label = args
    try:
        features, method = extract_hand_landmarks(img_path, _hands_primary, _hands_relaxed)
        return img_path, label, features, method
    except Exception as e:
        return img_path, label, None, f"exception: {e}"


def process_images_parallel(image_label_pairs, desc="A processar", num_workers=None):
    """Processa uma lista de (img_path, label) em paralelo com ProcessPoolExecutor.
    Cada processo tem os seus próprios detectors MediaPipe.
    Retorna (rows, failed, extraction_methods)."""
    rows = []
    failed = []
    extraction_methods = Counter()
    per_label_ok = Counter()
    per_label_fail = Counter()

    n_workers = num_workers or multiprocessing.cpu_count()

    with ProcessPoolExecutor(
        max_workers=n_workers,
        initializer=_worker_init
    ) as executor:
        futures = {executor.submit(_worker_process_image, pair): pair for pair in image_label_pairs}
        for future in tqdm(as_completed(futures), total=len(futures), desc=desc):
            img_path, label, features, method = future.result()
            if features is None:
                failed.append({"path": img_path, "label": label, "reason": method})
                per_label_fail[label] += 1
            else:
                rows.append(features + [label])
                extraction_methods[method] += 1
                per_label_ok[label] += 1

    return rows, failed, extraction_methods, per_label_ok, per_label_fail


def process_train_dataset(train_dir, num_workers=None):
    """Processa o dataset de treino em paralelo."""
    all_pairs = []
    class_totals = {}

    for label in CLASSES:
        class_dir = os.path.join(train_dir, label)
        if not os.path.isdir(class_dir):
            print(f"[AVISO] Pasta não encontrada: {class_dir}")
            continue

        image_files = sorted(
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )
        if MAX_IMAGES_PER_CLASS is not None:
            image_files = image_files[:MAX_IMAGES_PER_CLASS]

        class_totals[label] = len(image_files)
        for fname in image_files:
            all_pairs.append((os.path.join(class_dir, fname), label))

    rows, failed, extraction_methods, per_label_ok, per_label_fail = process_images_parallel(
        all_pairs, desc="Treino", num_workers=num_workers
    )

    stats = {}
    for label in class_totals:
        total = class_totals[label]
        ok = per_label_ok[label]
        bad = per_label_fail[label]
        stats[label] = {
            "total": total,
            "ok": ok,
            "failed": bad,
            "success_rate": round(100 * ok / total, 2) if total > 0 else 0.0
        }

    return rows, failed, stats, extraction_methods


def process_test_dataset(test_dir, num_workers=None):
    """Processa o dataset de teste em paralelo.
    Respeita MAX_IMAGES_PER_CLASS para consistência com o treino."""
    if not os.path.isdir(test_dir):
        print(f"[AVISO] Pasta de teste não encontrada: {test_dir}")
        return [], [], {}, Counter()

    test_files = collect_test_files(test_dir, max_per_class=MAX_IMAGES_PER_CLASS)
    print(f"[INFO] Ficheiros de teste encontrados: {len(test_files)}")

    rows, failed, extraction_methods, per_label_ok, per_label_fail = process_images_parallel(
        test_files, desc="Teste", num_workers=num_workers
    )

    test_stats = {}
    labels = sorted({label for _, label in test_files})
    for label in labels:
        ok = per_label_ok[label]
        bad = per_label_fail[label]
        total = ok + bad
        test_stats[label] = {
            "total": total,
            "ok": ok,
            "failed": bad,
            "success_rate": round(100 * ok / total, 2) if total > 0 else 0.0
        }

    return rows, failed, test_stats, extraction_methods


def count_images_in_directory(root_dir):
    total = 0
    for current_root, _, files in os.walk(root_dir):
        total += sum(
            1 for f in files
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        )
    return total


print("Imagens no treino:", count_images_in_directory(TRAIN_DIR) if os.path.isdir(TRAIN_DIR) else "diretório não encontrado")
print("Imagens no teste:", count_images_in_directory(TEST_DIR) if os.path.isdir(TEST_DIR) else "diretório não encontrado")
print(f"Núcleos disponíveis: {multiprocessing.cpu_count()} (NUM_WORKERS={NUM_WORKERS or multiprocessing.cpu_count()})") 


Imagens no treino: 102000
Imagens no teste: 2455
Núcleos disponíveis: 16 (NUM_WORKERS=16)


In [5]:
def save_failed_log(path, failed_items):
    with open(path, "w", encoding="utf-8") as f:
        for item in failed_items:
            if isinstance(item, dict):
                f.write(f"{item['path']} | label={item.get('label', '?')} | reason={item['reason']}\n")
            else:
                f.write(str(item) + "\n")


In [6]:
def main():
    os.makedirs("outputs", exist_ok=True)

    columns = build_columns()

    print(f"[INFO] TRAIN_DIR = {TRAIN_DIR}")
    print(f"[INFO] TEST_DIR  = {TEST_DIR}")
    print(f"[INFO] NUM_WORKERS = {NUM_WORKERS or multiprocessing.cpu_count()}")

    # =========================
    # TREINO
    # =========================
    print("\nA processar treino...")
    train_rows, train_failed, train_stats, train_methods = process_train_dataset(TRAIN_DIR, num_workers=NUM_WORKERS)
    df_train = pd.DataFrame(train_rows, columns=columns)
    df_train.to_csv(OUTPUT_TRAIN_CSV, index=False)

    # =========================
    # TESTE
    # =========================
    print("\nA processar teste...")
    test_rows, test_failed, test_stats, test_methods = process_test_dataset(TEST_DIR, num_workers=NUM_WORKERS)
    df_test = pd.DataFrame(test_rows, columns=columns)
    df_test.to_csv(OUTPUT_TEST_CSV, index=False)

    # =========================
    # RESUMO GLOBAL
    # =========================
    print("\n===== RESUMO =====")
    print(f"Treino: {len(df_train)} amostras guardadas em {OUTPUT_TRAIN_CSV}")
    print(f"Teste:  {len(df_test)} amostras guardadas em {OUTPUT_TEST_CSV}")
    print(f"Falhas treino: {len(train_failed)}")
    print(f"Falhas teste:  {len(test_failed)}")

    # =========================
    # STATS TREINO
    # =========================
    print("\n===== SUCESSO POR CLASSE (TREINO) =====")
    for label, info in train_stats.items():
        print(
            f"{label}: {info['ok']}/{info['total']} "
            f"({info['success_rate']}%) - falhas: {info['failed']}"
        )

    # =========================
    # STATS TESTE
    # =========================
    if test_stats:
        print("\n===== SUCESSO POR CLASSE (TESTE) =====")
        for label, info in test_stats.items():
            print(
                f"{label}: {info['ok']}/{info['total']} "
                f"({info['success_rate']}%) - falhas: {info['failed']}"
            )

    # =========================
    # GUARDAR STATS EM CSV
    # =========================
    pd.DataFrame(train_stats).T.to_csv(OUTPUT_TRAIN_STATS)
    if test_stats:
        pd.DataFrame(test_stats).T.to_csv(OUTPUT_TEST_STATS)

    # =========================
    # GUARDAR LOGS DE FALHAS
    # =========================
    if train_failed:
        save_failed_log("outputs/train_failed.txt", train_failed)
    if test_failed:
        save_failed_log("outputs/test_failed.txt", test_failed)

    # =========================
    # MÉTODOS USADOS
    # =========================
    print("\n===== MÉTODOS DE EXTRAÇÃO USADOS (TREINO) =====")
    for method, count in train_methods.most_common():
        print(f"{method}: {count}")

    if test_methods:
        print("\n===== MÉTODOS DE EXTRAÇÃO USADOS (TESTE) =====")
        for method, count in test_methods.most_common():
            print(f"{method}: {count}")


if __name__ == "__main__":
    main()


[INFO] TRAIN_DIR = asl/train
[INFO] TEST_DIR  = asl/test
[INFO] NUM_WORKERS = 16

A processar treino...
[AVISO] Pasta não encontrada: asl/train/J
[AVISO] Pasta não encontrada: asl/train/Z
[AVISO] Pasta não encontrada: asl/train/nothing


I0000 00:00:1779295137.288844   30087 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779295137.292153   30290 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Graphics (ADL GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
I0000 00:00:1779295137.297365   30087 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779295137.298856   30316 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Graphics (ADL GT2)
I0000 00:00:1779295137.310819   30156 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779295137.310931   30088 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1779295137.314522   30385 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.3), renderer: Mesa Intel(R) Graphics (ADL GT2)
I0000 00:00:1779

KeyboardInterrupt: 